# ДЗ2, опыт24: быстрый рецепт при равных предъявлениях — LR и две абляции

**Гипотеза:** A(4+4/d256/h4/SwiGLU680,12.09M) +AMP FP16/batch128/bucketing достигает ≥18% exec-match на старых991 за примерно1/5 прежнего времени при800k предъявлениях. Общий план: **52 797 347 полезных target-токенов**,38 706 084 source; PAD исключён, target включает EOS. Это гипотеза, результатов24 ещё нет.

|Запуск|Peak LR|Batch/updates|Изменение|
|---|---:|---|---|
|R1|1e-3|128×6250|Базовый быстрый рецепт|
|R2|2e-3|128×6250|Только LR|
|R3|4e-3|128×6250|Только LR|
|R4|ЛучшийR1–R3|128×6250|Dropout0|
|R5|ЛучшийR1–R3|128×6250|Токены init std1/√d при прежнем множителе√d; позиции прежние;dropout.1|
|R6 условно|ЛучшийLR/2|32×25000|Если лучшийR1–R3 даёт<18% на старых991|

В каждом запуске800000 предъявлений; R6 получает тот же плоский список пар. Общая инициализация с seed42; R5 делит только исходную token-table на√d. AdamW wd.01/clip1; LR warmup5%(ceil:313/6250,1250/25000), затем cosine до10% пика. Прогрев LR входит в6250, дополнительных train-батчей нет. AMP scaler init_scale1024 одинаков во всех запусках; unscale доclip. Пропуск AMP update/nonfinite делает запуск непригодным для решения; дополнительные предъявления для компенсации не добавляются.

Train — прежние500k/BPE8k, без фильтрации reference. Новый heldout4k — первые4096 уникальных числовых hash-heldout из shards3–5; исключеныtest/old/fresh. Fresh1024 защищён и не оценивается. Основная метрика — исполнение reference, не математическое исправление reference.

Рецепт принимается: лучший запуск≥18% на991 и потеря≤1.5п.п. против22 на4k. Абляция принимается при потере≤0.5п.п. против лучшегоLR, явный выигрыш≥1п.п. Точное правило выбора — exec4k, затем CE4k, затем ID при полном равенстве. McNemar выводится отдельно. Если проходит толькоR6, гипотеза batch128 не подтверждена.

Ориентир бюджета: около90 мин, сR6 около115; фактические времена записываются. Запуски не сокращаются ради таймера. МодельB/C, новые train-данные/фильтрация, beam иsampling сюда не входят.

In [ ]:
import time
if '_RECIPE24_FIRST_CELL' not in globals():
    _RECIPE24_FIRST_CELL=time.monotonic()
%pip install -q tokenizers==0.23.2 pyarrow sympy

## Colab и файлы

Выбрать одну T4: Runtime → Change runtime type → T4 GPU. Для нового опыта достаточно компактного **24-start-from-22-model-only.zip** (около 49 МБ): проверенные веса22/100k, tokenizer и manifest. Исходный ZIP22 занимает около 391 МБ. Его или PT22/100k + tokenizer JSON по-прежнему можно использовать. ZIP21 или PT21/100k можно добавить для второго контроля.

В ячейке входных файлов выбрать `INPUT_MODE`:

- `auto` (по умолчанию): если входные файлы уже проверены в этой сессии, использовать их; иначе открыть загрузку с компьютера. Выбрать компактный ZIP.
- `drive`: один раз положить компактный ZIP в свою папку Google Drive `nlp-hw02`, указать его в `INPUT_PATHS`. Colab запросит доступ к Drive и скопирует только указанные файлы на локальный диск. При следующем runtime повторная загрузка с компьютера не нужна.
- `paths`: указать файлы, уже находящиеся на диске Colab, например после загрузки через боковую панель Files.
- `upload`: принудительно открыть загрузку, например для добавления контроля21; `reuse`: использовать только уже подготовленную папку результатов.

**Resume:** в новой сессии указать последний ZIP24 через `drive`/`paths` либо загрузить его. Он содержит tokenizer, control-веса, общий initial-state, план батчей, heldout и завершённые результаты. Для resume нужен именно ZIP24 с результатами; компактный стартовый ZIP содержит только вход для нового опыта. Код рецепта и формат resume сохранены: архивы прежней версии24 совместимы. Для добавления контроля21 указать его вместе с ZIP24.

Внутри запуска step-resume отсутствует: незавершённый train стартует с общей инициализации. Если train уже завершён, продолжается оценка без повторного обучения. ZIP обновляется после train и evaluation каждого запуска. После прерывания выполнить последнюю ячейку: она умеет скачать ZIP или сохранить его в Drive. Optimizer/scaler state не сохраняются.


In [ ]:
import os,sys,json,hashlib,gc,subprocess,importlib
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='1'
MODULE_DIR=Path('hw02-recipe24-modules')
MODULE_DIR.mkdir(exist_ok=True)
MODULE_SOURCES={'recipe24_core.py': 'import ast,contextlib,gc,hashlib,json,math,random,time\n\nfrom array import array\n\nfrom pathlib import Path\n\nfrom collections import Counter\n\nimport torch\n\nfrom torch import nn\n\nfrom torch.nn import functional as F\n\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\n\nfrom torch.nn.utils.rnn import pad_sequence\n\nfrom tokenizers import Tokenizer\n\nVOCAB=8192\n\nMAX_POSITIONS=2048\n\nGENERATION_CAP=512\n\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError(\'Position overflow\')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\ndef sync(device):\n if device.type==\'cuda\':torch.cuda.synchronize(device)\n\ndef init_tokenizer(path):\n tokenizer=Tokenizer.from_file(str(path))\n if tokenizer.get_vocab_size()!=VOCAB-TOKEN_OFFSET:raise ValueError(\'Unexpected vocabulary\')\n visible=list(range(33,127))+list(range(161,173))+list(range(174,256));bchars={b:chr(b)for b in visible}\n for i,b in enumerate(b for b in range(256)if b not in bchars):bchars[b]=chr(256+i)\n charbytes={c:b for b,c in bchars.items()}\n tokenbytes=[b\'\',b\'\',b\'\']+[bytes(charbytes[c]for c in tokenizer.id_to_token(i))for i in range(VOCAB-3)]\n return tokenizer,tokenbytes\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b\'\'.join(chunks)\n\ndef source_batch(tokenizer,questions,device):\n return pad_sequence([torch.tensor([BOS]+encode(tokenizer,q)+[EOS])for q in questions],batch_first=True,padding_value=PAD).to(device)\n\ndef collate_cache(cache,indices,device):\n src=pad_sequence([torch.tensor(cache[i][0],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n tgt=pad_sequence([torch.tensor([BOS]+list(cache[i][1]),dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n labels=pad_sequence([torch.tensor(list(cache[i][1])+[EOS],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n counts={\'source_tokens\':sum(len(cache[i][0])for i in indices),\'target_tokens\':sum(len(cache[i][1])+1 for i in indices),\n  \'source_slots\':src.numel(),\'target_slots\':labels.numel(),\'pairs\':len(indices)}\n return src.to(device),tgt.to(device),labels.to(device),counts\n\ndef bucket_plan(cache,batch,steps,seed=230042,chunk_pairs=4096):\n if chunk_pairs%batch:raise ValueError(\'Chunk must be divisible by batch\')\n rng=random.Random(seed);plans=[]\n while len(plans)<steps:\n  chunk=rng.sample(range(len(cache)),min(chunk_pairs,len(cache)))\n  chunk.sort(key=lambda i:(len(cache[i][0])+len(cache[i][1])+1,len(cache[i][0]),len(cache[i][1])))\n  batches=[chunk[i:i+batch]for i in range(0,len(chunk)-batch+1,batch)]\n  if not batches:raise ValueError(\'Too few data for this batch size\')\n  rng.shuffle(batches);plans.extend(batches)\n return plans[:steps]\n\ndef sdpa_flags():\n return {k:bool(getattr(torch.backends.cuda,k+\'_sdp_enabled\')())for k in [\'flash\',\'mem_efficient\',\'math\',\'cudnn\']if hasattr(torch.backends.cuda,k+\'_sdp_enabled\')}\n\ndef profile_sdpa(call,device):\n if device.type!=\'cuda\':return {\'status\':\'CPU_smoke\',\'operators\':[]}\n with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA])as prof:\n  call();sync(device)\n ops=sorted({e.key for e in prof.key_averages()if \'scaled_dot_product\'in e.key})\n return {\'status\':\'observed\',\'operators\':ops,\'scope\':\'one inference preflight outside timers; other shapes/backward may use a different backend\'}\n\ndef fused_adamw_available(device):\n if device.type!=\'cuda\':return False\n try:\n  p=nn.Parameter(torch.ones(1,device=device));opt=torch.optim.AdamW([p],lr=1e-4,fused=True);p.grad=torch.ones_like(p);opt.step();sync(device)\n  del opt,p;return True\n except (TypeError,RuntimeError):return False\n\nclass CachedDecoder:\n    @torch.inference_mode()\n    def __init__(self, model, src, capacity):\n        model.eval()\n        if not 0 < capacity <= model.positions.num_embeddings:\n            raise ValueError(\'Invalid cache capacity\')\n        self.model, self.capacity, self.position = model, capacity, 0\n        self.batch_size = len(src)\n        self.width = model.tokens.embedding_dim\n        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads\n        self.head_dim = self.width // self.heads\n        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))\n        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).\n        self.source_allowed = src.ne(PAD)[:, None, None, :]\n        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)\n        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))\n        self.layers = []\n        for layer in model.transformer.decoder.layers:\n            if not layer.norm_first:\n                raise ValueError(\'This experiment requires Pre-LN decoder layers\')\n            for attention in (layer.self_attn, layer.multihead_attn):\n                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads\n                        or attention.bias_k is not None or attention.bias_v is not None\n                        or attention.add_zero_attn):\n                    raise ValueError(\'Unsupported attention configuration\')\n            attention = layer.multihead_attn\n            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],\n                                    attention.in_proj_bias[self.width:])\n            cross_k, cross_v = projected_kv.chunk(2, dim=-1)\n            shape = (len(src), self.heads, capacity, self.head_dim)\n            self.layers.append({\n                \'self_k\': memory.new_empty(shape), \'self_v\': memory.new_empty(shape),\n                \'cross_k\': self.split_heads(cross_k), \'cross_v\': self.split_heads(cross_v),\n            })\n\n    def split_heads(self, values):\n        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)\n\n    def attend(self, attention, q, k, v, allowed):\n        # One query at the latest position; all supplied keys are from its past/current.\n        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.\n        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,\n                                                dropout_p=0.0, is_causal=False)\n        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)\n        return attention.out_proj(output)\n\n    @torch.inference_mode()\n    def step(self, token_ids):\n        if self.position >= self.capacity:\n            raise ValueError(\'Cache capacity exceeded\')\n        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:\n            raise ValueError(\'One token per batch row is required\')\n        t = self.position\n        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)\n        x = x + self.position_vectors[t][None, None, :]\n        self.valid[:, t] = token_ids.ne(PAD)\n        self_allowed = self.valid[:, None, None, :t + 1]\n        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):\n            attention = layer.self_attn\n            q, k, v = F.linear(layer.norm1(x), attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)\n            cache[\'self_k\'][:, :, t:t + 1] = self.split_heads(k)\n            cache[\'self_v\'][:, :, t:t + 1] = self.split_heads(v)\n            self_output = self.attend(attention, self.split_heads(q),\n                cache[\'self_k\'][:, :, :t + 1], cache[\'self_v\'][:, :, :t + 1], self_allowed)\n            x = x + layer.dropout1(self_output)\n\n            attention = layer.multihead_attn\n            q = F.linear(layer.norm2(x), attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])\n            cross_output = self.attend(attention, self.split_heads(q),\n                                      cache[\'cross_k\'], cache[\'cross_v\'], self.source_allowed)\n            x = x + layer.dropout2(cross_output)\n            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x)))))\n            x = x + layer.dropout3(feed_forward)\n\n        if self.model.transformer.decoder.norm is not None:\n            x = self.model.transformer.decoder.norm(x)\n        self.position += 1\n        return self.model.head(x[:, 0])\n\ndef code_properties(code):\n    def scoped_nodes(node):\n        stack = [node]\n        while stack:\n            current = stack.pop()\n            yield current\n            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):\n                stack.extend(ast.iter_child_nodes(current))\n    try:\n        tree = ast.parse(code)\n        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'solve\']\n        has_return = any(isinstance(n, ast.Return) and n.value is not None\n            for f in functions for stmt in f.body for n in scoped_nodes(stmt))\n        return {\'parses_as_python\': True, \'defines_solve\': bool(functions), \'solve_returns_value\': has_return,\n                \'syntax_error_type\': None, \'syntax_error_message\': None, \'indentation_error\': False}\n    except (SyntaxError, ValueError, RecursionError) as error:\n        return {\'parses_as_python\': False, \'defines_solve\': False, \'solve_returns_value\': False,\n                \'syntax_error_type\': type(error).__name__, \'syntax_error_message\': str(error),\n                \'indentation_error\': isinstance(error, IndentationError)}\n\nclass BudgetExpired(Exception):pass\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired(\'Session budget reached; completed run boundaries are preserved\')\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n@torch.inference_mode()\ndef generate_preallocated(model,src,cap=GENERATION_CAP,deadline=None,check_every=16):\n """Only output storage and termination checks change; same encoder/KV/argmax."""\n cache=CachedDecoder(model,src,cap)\n output=torch.full((len(src),cap),PAD,device=src.device,dtype=torch.long)\n token=torch.full((len(src),),BOS,device=src.device,dtype=torch.long)\n done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n for step in range(cap):\n  if step%check_every==0:check_deadline(deadline)\n  logits=cache.step(token);logits[:,[PAD,BOS]]=-torch.inf\n  token=logits.argmax(-1).masked_fill(done,PAD);output[:,step]=token;done|=token.eq(EOS)\n  # Only one device->CPU synchronization per 16 steps, plus final output transfer.\n  # Rows already at EOS remain PAD; active_tokens removes these extra slots.\n  if (step+1)%check_every==0 and bool(done.all()):break\n return output[:,:cache.position],cache.position\n\ndef restore_order(ordered_values,original_indices,size):\n if len(ordered_values)!=size or sorted(original_indices)!=list(range(size)):\n  raise ValueError(\'Not a complete permutation\')\n restored=[None]*size\n for value,i in zip(ordered_values,original_indices):restored[i]=value\n return restored\n\ndef inspect_output(row,ids,tokenizer,tokenbytes):\n active=active_tokens(ids);raw=decode_bytes(tokenbytes,active)\n try:code=raw.decode(\'utf-8\');valid=True\n except UnicodeDecodeError:code=raw.decode(\'utf-8\',errors=\'replace\');valid=False\n example={\'question\':row[\'question\'],\'generated_code\':code,\'generated_token_ids\':active,\n  \'generated_tokens_including_eos\':len(active),\'generated_utf8_bytes\':len(raw),\'valid_utf8\':valid,\n  \'eos_emitted\':EOS in active,\'reached_token_cap\':len(active)==GENERATION_CAP,\n  \'token_cap_without_eos\':len(active)==GENERATION_CAP and EOS not in active,\n  \'byte_budget_stop\':False,\'stop_reason\':\'eos\'if EOS in active else\'token_cap\',**code_properties(code)}\n if \'code\'in row:\n  gold=encode(tokenizer,row[\'code\'])+[EOS]\n  example.update(reference_code=row[\'code\'],reference_utf8_bytes=len(row[\'code\'].encode()),\n   reference_bpe_tokens_including_eos=len(gold),exact_code_match=valid and code==row[\'code\'],exact_tokens_with_eos=active==gold)\n return example\n\ndef sha_file(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\')as f:\n  for chunk in iter(lambda:f.read(1024**2),b\'\'):h.update(chunk)\n return h.hexdigest()\n\ndef write_json(path,value):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+\'.tmp\')\n tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\');tmp.replace(path)\n\nMODEL_SPEC=dict(d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680)\n', 'recipe24_exec.py': '"""Bounded numeric-program execution for this public dataset; no arbitrary imports/I/O."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef execute_task(payload):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    tree,status=filter_program(code)\n    if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:signal.setitimer(signal.ITIMER_REAL,0)\n\ndef create_pool(workers,memory_bytes):\n    return mp.get_context(\'spawn\').Pool(workers,initializer=worker_init,\n        initargs=(memory_bytes,),maxtasksperchild=256)\n\ndef evaluate_programs(codes,workers=2,wall_limit=1.0,cpu_limit=2.0,\n                      parent_timeout=3.0,memory_bytes=768*1024**2,deadline=None,progress=None):\n    """Bounded inflight tasks; every async result has a parent deadline.\n    A killed worker loses its task even if Pool replaces it: terminate/recreate\n    the WHOLE pool, retain completed results and retry collateral pending tasks.\n    Pool.map is deliberately never used. Output order and duplicates preserved.\n    """\n    if parent_timeout<=wall_limit:raise ValueError(\'Parent deadline must exceed program deadline\')\n    unique=list(dict.fromkeys(codes));todo=deque(range(len(unique)));answers={};pending={}\n    pool=None;restarts=0;start=time.monotonic();cancelled=False;last_progress=-1\n    try:\n        while todo or pending:\n            if deadline is not None and time.monotonic()>=deadline:\n                cancelled=True;break\n            if pool is None:pool=create_pool(workers,memory_bytes)\n            while todo and len(pending)<workers:\n                i=todo.popleft();tree,status=filter_program(unique[i])\n                if status:answers[i]={\'status\':status};continue\n                pending[i]=(pool.apply_async(execute_task,((unique[i],wall_limit,cpu_limit),)),time.monotonic())\n            for i,(future,submitted)in list(pending.items()):\n                if future.ready():\n                    try:answers[i]=future.get(timeout=0)\n                    except Exception as e:answers[i]={\'status\':\'worker_error\',\'error_type\':type(e).__name__}\n                    del pending[i]\n            timed=[i for i,(_,submitted)in pending.items()if time.monotonic()-submitted>=parent_timeout]\n            if timed:\n                for i in timed:answers[i]={\'status\':\'timeout_or_worker_exit\'}\n                retry=[i for i in pending if i not in timed]\n                pool.terminate();pool.join();pool=None;restarts+=1;pending.clear()\n                todo.extendleft(reversed(retry))\n            if progress and len(answers)!=last_progress and (len(answers)%1000==0 or len(answers)==len(unique)):\n                progress(len(answers),len(unique));last_progress=len(answers)\n            if pending:time.sleep(.001)\n        if cancelled:\n            for i in todo:answers[i]={\'status\':\'budget_not_evaluated\'}\n            for i in pending:answers[i]={\'status\':\'budget_not_evaluated\'}\n    finally:\n        if pool is not None:pool.terminate();pool.join()\n    by_code={c:answers.get(i,{\'status\':\'budget_not_evaluated\'})for i,c in enumerate(unique)}\n    rows=[dict(by_code[c])for c in codes]\n    return {\'rows\':rows,\'summary\':{\'programs\':len(codes),\'unique_programs\':len(unique),\n        \'statuses\':dict(Counter(r[\'status\']for r in rows)),\'pool_restarts\':restarts,\n        \'seconds\':time.monotonic()-start,\'complete\':not cancelled}}\n\ndef compare_exec(reference_results,generation_results,rel_tol=1e-6,abs_tol=1e-6):\n    refs=reference_results[\'rows\']if isinstance(reference_results,dict)else reference_results\n    gens=generation_results[\'rows\']if isinstance(generation_results,dict)else generation_results\n    if len(refs)!=len(gens):raise ValueError(\'Order/length mismatch\')\n    eligible=[i for i,r in enumerate(refs)if r[\'status\']==\'numeric\']\n    matches=[i for i in eligible if gens[i][\'status\']==\'numeric\'and math.isclose(\n        gens[i][\'value\'],refs[i][\'value\'],rel_tol=rel_tol,abs_tol=abs_tol)]\n    return {\'denominator\':len(eligible),\'matches\':len(matches),\n        \'accuracy\':len(matches)/len(eligible)if eligible else None,\n        \'eligible_indices\':eligible,\'match_indices\':matches,\n        \'reference_statuses\':dict(Counter(r[\'status\']for r in refs)),\n        \'generation_statuses\':dict(Counter(r[\'status\']for r in gens)),\n        \'rel_tol\':rel_tol,\'abs_tol\':abs_tol}\n', 'recipe24_data.py': 'import hashlib,json,random,time\nfrom pathlib import Path\nfrom array import array\nfrom urllib.request import urlretrieve\nimport pyarrow.parquet as pq\nfrom recipe24_core import sha_file,write_json,encode,BOS,EOS,check_deadline\nREVISION=\'b2724689ff079e9c22efd548fe134bfb29036dbb\'\nSHARDS=[\n {\'path\':\'data/train/part-00000.parquet\',\'bytes\':26169796,\'sha256\':\'91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec\'},\n {\'path\':\'data/train/part-00001.parquet\',\'bytes\':26166175,\'sha256\':\'5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068\'},\n {\'path\':\'data/train/part-00002.parquet\',\'bytes\':26162244,\'sha256\':\'c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de\'}]\nTRAIN_HASH=\'61483f4dc6970ff5f2b1afffa45d57391dcfc01ab6ec4f140c6e6b6c792c8c38\'\nLOCATIONS_HASH=\'c2332a54a923ffa85da9ac91ec0a39a2e4ead7f7c2e328abecef702e1d306ba7\'\nHELDOUT_HASH=\'e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743\'\nTEST_HASH=\'8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a\'\n\ndef data_hash(rows):return hashlib.sha256(json.dumps(rows,ensure_ascii=False,sort_keys=True).encode()).hexdigest()\ndef is_validation(question):return int.from_bytes(hashlib.sha256(f\'42:{question}\'.encode()).digest()[:8],\'big\')%10==0\ndef eligible(row):return len(row[\'question\'].encode())<=512 and len(row[\'code\'].encode())<512\n\ndef download_checked(spec,cache,deadline=None):\n check_deadline(deadline);cache=Path(cache);cache.mkdir(parents=True,exist_ok=True);target=cache/Path(spec[\'path\']).name\n if not target.exists():\n  tmp=target.with_suffix(\'.download\')\n  # Network calls have a bounded socket timeout; deadline checked again afterward.\n  import socket\n  socket.setdefaulttimeout(30)\n  urlretrieve(f\'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{REVISION}/{spec["path"]}\',tmp)\n  if sha_file(tmp)!=spec[\'sha256\']:raise ValueError(\'Downloaded SHA mismatch\')\n  tmp.replace(target)\n if sha_file(target)!=spec[\'sha256\']:raise ValueError(\'Cached SHA mismatch\')\n if \'bytes\'in spec and target.stat().st_size!=spec[\'bytes\']:raise ValueError(\'Size mismatch\')\n check_deadline(deadline);return target\n\ndef prepare_data(cache,tokenizer,deadline=None,progress=print):\n started=time.monotonic()\n test_file=download_checked({\'path\':\'data/test/test.parquet\',\'sha256\':TEST_HASH},cache,deadline)\n test=[{\'question\':q}for q in pq.read_table(test_file,columns=[\'question\'])[\'question\'].to_pylist()]\n assert len(test)==1919;test_questions={r[\'question\']for r in test}\n train=[];locations=[];validation=None;heldout_pool=[];heldout_questions=set();paths=[];fresh=[];fresh_questions=set()\n # Exact21/22 selection. No reference-exec filtering in this throughput experiment.\n for shard_index,spec in enumerate(SHARDS):\n  path=download_checked(spec,cache,deadline);paths.append(path);offset=0\n  for batch in pq.ParquetFile(path).iter_batches(batch_size=4096):\n   check_deadline(deadline);rows=batch.to_pylist()\n   if validation is None:\n    validation=[r for r in rows if eligible(r)and is_validation(r[\'question\'])][:64]\n    assert len(validation)==64;heldout_questions.update(r[\'question\']for r in validation)\n   for i,row in enumerate(rows):\n    if not eligible(row)or row[\'question\']in test_questions:continue\n    if is_validation(row[\'question\']):\n     if shard_index==0 and row[\'question\']not in heldout_questions and len(heldout_pool)<960:\n      heldout_pool.append(row);heldout_questions.add(row[\'question\'])\n     if shard_index>0 and row[\'question\']not in heldout_questions and row[\'question\']not in fresh_questions and len(fresh)<1024:\n      fresh.append(row);fresh_questions.add(row[\'question\'])\n     continue\n    train.append(row);locations.append((shard_index,offset+i))\n    if len(train)==500000:break\n   if len(train)==500000:break\n   offset+=len(rows)\n  if progress:progress(f\'Data shard{shard_index}: selected {len(train)} /500000\')\n  if len(train)==500000:break\n heldout=validation+heldout_pool\n assert len(train)==500000 and len(heldout)==1024 and len(heldout_questions)==1024\n assert data_hash(train)==TRAIN_HASH and data_hash(locations)==LOCATIONS_HASH and data_hash(heldout)==HELDOUT_HASH\n assert not any(r[\'question\']in heldout_questions or r[\'question\']in test_questions for r in train)\n summary={\'revision\':REVISION,\'train_rows\':len(train),\'heldout_rows\':len(heldout),\'test_rows\':len(test),\n  \'train_sha256\':TRAIN_HASH,\'locations_sha256\':LOCATIONS_HASH,\'heldout_sha256\':HELDOUT_HASH,\'shards\':SHARDS,\n  \'selection_seconds\':time.monotonic()-started,\'filter_reference_non_number\':False,\'fresh_heldout_not_evaluated\':True}\n # encode_batch keeps the unchanged tokenization but avoids a Python call per string.\n token_cache=[];encoding_started=time.monotonic()\n for lo in range(0,len(train),2048):\n  check_deadline(deadline);part=train[lo:lo+2048]\n  questions=tokenizer.encode_batch([r[\'question\']for r in part],add_special_tokens=False)\n  codes=tokenizer.encode_batch([r[\'code\']for r in part],add_special_tokens=False)\n  token_cache.extend((array(\'H\',[BOS]+[i+3 for i in q.ids]+[EOS]),array(\'H\',[i+3 for i in c.ids]))for q,c in zip(questions,codes))\n summary[\'bpe_cache_seconds\']=time.monotonic()-encoding_started\n summary[\'source_tokens\']=sum(len(x[0])for x in token_cache);summary[\'target_tokens\']=sum(len(x[1])+1 for x in token_cache)\n assert len(fresh)==1024 and data_hash(fresh)==\'eaf5f7c00559d6b1e5650a0dd879e5b4a214931ee803f8c1f6efd956c54971b2\'\n summary[\'fresh_protection\']={\'questions\':1024,\'pairs_sha256\':data_hash(fresh),\'usage\':\'exclusion set only; not encoded/evaluated/saved/modified\'}\n return token_cache,heldout,test,paths,summary,fresh_questions\n\nNEW_SHARDS=[{\'path\': \'data/train/part-00003.parquet\', \'bytes\': 26167481, \'sha256\': \'b2b63db46b005e229aa5e91f1010adc763ca9f4b806561b328432fbc9e5c11e9\'}, {\'path\': \'data/train/part-00004.parquet\', \'bytes\': 26185069, \'sha256\': \'272ac87a686134a3e3ba6488985879bd8ea567dcaba1e1145aae0cb724a0afe5\'}, {\'path\': \'data/train/part-00005.parquet\', \'bytes\': 26168730, \'sha256\': \'89687321a1df7338e543a7d8c46d7bc0bd6596d855f9d3e43995e2a9cff858b7\'}]\n\ndef select_candidates(chunks, forbidden, evaluate, count=4096, deadline=None):\n    """First numeric, unique-question rows in physical shard/row order.\n    Duplicate candidates are retained until execution, so an invalid first\n    reference cannot hide a later valid row with the same question.\n    """\n    from collections import Counter\n    selected, values, locations = [], [], []\n    selected_questions = set()\n    audit = Counter()\n    for shard, offset, rows in chunks:\n        check_deadline(deadline)\n        candidates = []\n        for i, row in enumerate(rows):\n            audit[\'raw_rows_seen\'] += 1\n            if not eligible(row) or not is_validation(row[\'question\']):\n                continue\n            if row[\'question\'] in forbidden or row[\'question\'] in selected_questions:\n                continue\n            candidates.append((offset+i, row))\n        if not candidates:\n            continue\n        execution = evaluate([r[\'code\'] for _, r in candidates], deadline=deadline)\n        if not execution[\'summary\'][\'complete\']:\n            raise RuntimeError(\'Incomplete heldout reference execution\')\n        audit[\'reference_candidates_executed\'] += len(candidates)\n        audit[\'pool_restarts\'] += execution[\'summary\'][\'pool_restarts\']\n        for (index, row), value in zip(candidates, execution[\'rows\']):\n            audit[\'reference_\'+value[\'status\']] += 1\n            if value[\'status\'] != \'numeric\' or row[\'question\'] in selected_questions:\n                continue\n            selected.append(row); values.append(value); locations.append((shard, index))\n            selected_questions.add(row[\'question\'])\n            if len(selected) == count:\n                return selected, {\'rows\': values}, {\n                    \'selection\': \'first unique numeric eligible hash-heldout rows; physical shard3..5/row order\',\n                    \'count\': count, \'pairs_sha256\': data_hash(selected),\n                    \'locations_sha256\': data_hash(locations), \'locations\': locations,\n                    \'audit\': dict(audit), \'question_overlap_forbidden\': 0}\n    raise RuntimeError(f\'Only {len(selected)} eligible numeric unique heldout rows\')\n\ndef prepare_heldout4k(cache, forbidden, evaluate, deadline=None):\n    def chunks():\n        for shard, spec in enumerate(NEW_SHARDS, 3):\n            path = download_checked(spec, cache, deadline)\n            offset = 0\n            for batch in pq.ParquetFile(path).iter_batches(batch_size=4096):\n                rows = batch.to_pylist()\n                yield shard, offset, rows\n                offset += len(rows)\n    return select_candidates(chunks(), forbidden, evaluate, deadline=deadline)\n', 'recipe24_run.py': '"""Experiment24: equal pair presentations, fixed data, LR search and two ablations."""\nimport gc, hashlib, json, math, os, time, zipfile\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport recipe24_core as core\nimport recipe24_exec as numeric\n\nSEED=42\nPLAN_SEED=230042\nPAIR_PRESENTATIONS=800_000\nPROTOCOL={\n    \'experiment\':24,\'revision\':\'b2724689ff079e9c22efd548fe134bfb29036dbb\',\n    \'model\':core.MODEL_SPEC,\'train_pairs\':500000,\'pair_presentations\':PAIR_PRESENTATIONS,\n    \'batch128_steps\':6250,\'fallback_batch32_steps\':25000,\'plan_seed\':PLAN_SEED,\'init_seed\':SEED,\n    \'chunk_pairs\':4096,\'peak_LRs\':[.001,.002,.004],\'warmup_fraction\':.05,\'warmup_rounding\':\'ceil\',\n    \'cosine_floor_fraction\':.1,\'dropout\':.1,\'weight_decay\':.01,\'clip\':1.,\n    \'AMP\':\'FP16 autocast + GradScaler; unscale before clip\',\'initial_scale\':1024,\n    \'optimizer\':\'fused AdamW when supported; same mode for all launches\',\n    \'R4\':\'best LR, dropout0, otherwise base recipe\',\n    \'R5\':\'best LR, dropout.1; same initial token draws divided by sqrt(d), positions unchanged\',\n    \'R6\':\'if best R1..R3 old991<.18; batch32, best peak LR/2, identical flattened pair plan\',\n    \'generation\':\'FP16 sorted batch512; preallocated output; CPU EOS check16; greedy cap512\',\n    \'CE\':\'FP32 eval, token-weighted sum/PAD excluded, EOS included, batch32\',\n    \'reference_length_bins\':\'content BPE tokens without EOS: [0,60),[60,90),[90,120),[120,inf)\',\n    \'metric\':\'executed reference solve; finite numeric; bool->int; math.isclose rel=abs=1e-6\',\n    \'accept_recipe\':{\'old991_min\':.18,\'heldout4k_drop_from22_max\':.015},\n    \'accept_ablation\':{\'heldout4k_drop_from_best_LR_max\':.005,\'clear_gain\':.01},\n    \'expected_main_minutes\':90,\'expected_with_R6_minutes\':115,\n    \'saved22_chain_training_seconds\':4007.492226436997,\n    \'resume\':\'completed training/evaluation receipts; interrupted training restarts from common initialization\',\n    \'quality_eligibility\':\'all planned presentations, no nonfinite loss/grad and zero AMP skipped updates\',\n    \'fresh1024\':\'protected exclusion set only, never encoded/evaluated/modified\',\n    \'train_filter_reference\':False,\n    \'heldout4k_pairs_sha256\':\'6b0d0d559a56921faac1842e3508889225af1fe1ad8e04de378840bc4df8b047\',\n    \'batch_plan_file_sha256\':\'48734ce4d9621e0adf464e5fad25bfc625772d7a222baeada107ea92c0a13628\',\n    \'expected_useful_source_tokens\':38706084,\'expected_useful_target_tokens\':52797347}\n\ndef protocol_hash():\n    return hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()\n\ndef save_model_state(path, model):\n    """Every PT contains only a dict of FP32 CPU model tensors. No optimizer/RNG pickle."""\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    state={k:v.detach().cpu().float().contiguous() for k,v in model.state_dict().items()}\n    tmp=path.with_suffix(\'.pt.tmp\');torch.save(state,tmp);tmp.replace(path)\n    return core.sha_file(path)\n\ndef state_digest(state):\n    h=hashlib.sha256()\n    for k,v in sorted(state.items()):\n        a=v.detach().cpu().contiguous().numpy();h.update(k.encode());h.update(str(a.shape).encode());h.update(a.tobytes())\n    return h.hexdigest()\n\ndef set_seed(seed=SEED):\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef make_initial(path, spec=None):\n    set_seed();model=core.SimpleTransformer(**(spec or core.MODEL_SPEC))\n    digest=state_digest(model.state_dict());save_model_state(path,model);del model\n    return digest\n\ndef load_model(path, device, dropout=.1, scaled_tokens=False, spec=None):\n    model=core.SimpleTransformer(**(spec or core.MODEL_SPEC),dropout=dropout)\n    state=torch.load(path,map_location=\'cpu\',weights_only=True)\n    model.load_state_dict(state,strict=True)\n    if scaled_tokens:\n        with torch.no_grad():model.tokens.weight.div_(math.sqrt(model.width))\n    del state\n    return model.to(device)\n\ndef lr_for_update(update,total,peak):\n    if not 1<=update<=total:raise ValueError(\'Update indexing is1..T\')\n    warmup=math.ceil(.05*total)\n    if update<=warmup:return peak*update/warmup\n    progress=(update-warmup)/(total-warmup)\n    return peak*(.1+.45*(1+math.cos(math.pi*progress)))\n\ndef run_spec(run_id, peak, batch=128, dropout=.1, scaled=False):\n    return {\'id\':run_id,\'peak_lr\':peak,\'batch\':batch,\'updates\':PAIR_PRESENTATIONS//batch,\n            \'dropout\':dropout,\'scaled_token_init\':scaled}\n\ndef train_one(config, initial_path, cache, common_plan, device, outdir,\n              fused=True, spec=None, amp=True, log_every=250, stop_after=None, amp_on_cpu=False):\n    """No extra warmup presentations. The LR warmup is inside the fixed plan.\n    Skipped AMP updates never cause extra presentations; the run is ineligible\n    for decisions if even one skip/nonfinite step occurred.\n    """\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    plan=common_plan.reshape(-1,config[\'batch\'])\n    assert plan.shape[0]==config[\'updates\'] and plan.size==config[\'updates\']*config[\'batch\']\n    model=opt=scaler=None;started=time.perf_counter();train_start=train_end=None;counts=Counter();history=[]\n    summary={\'config\':config,\'status\':\'training\',\'steps_presented\':0,\'optimizer_updates\':0,\n             \'AMP_skips\':0,\'finite_loss_and_grads\':True,\'fused_adamw\':fused,\n             \'warmup_steps\':math.ceil(.05*config[\'updates\']),\n             \'initial_checkpoint_sha256\':core.sha_file(initial_path)}\n    try:\n        gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n        model=load_model(initial_path,device,dropout=config[\'dropout\'],\n                         scaled_tokens=config[\'scaled_token_init\'],spec=spec)\n        summary[\'initial_model_tensor_sha256\']=state_digest(model.state_dict())\n        model.train();set_seed()\n        opt=torch.optim.AdamW(model.parameters(),lr=config[\'peak_lr\'],weight_decay=.01,\n                              **({\'fused\':True} if fused else {}))\n        amp_enabled=amp and (device.type==\'cuda\' or amp_on_cpu)\n        summary[\'precision\']=\'AMP_FP16\' if amp_enabled else \'FP32_CPU_smoke\'\n        scaler=torch.amp.GradScaler(device.type,enabled=amp_enabled,init_scale=1024)\n        if amp_enabled and not hasattr(scaler,\'_found_inf_per_device\'):\n            raise RuntimeError(\'Exact AMP skip diagnostic unavailable\')\n        core.sync(device);train_start=time.perf_counter()\n        for lo in range(0,len(plan),25):\n            records=[]\n            for j in range(lo,min(lo+25,len(plan))):\n                src,tgt,labels,batch_counts=core.collate_cache(cache,plan[j],device)\n                opt.zero_grad(set_to_none=True)\n                lr=lr_for_update(j+1,len(plan),config[\'peak_lr\'])\n                for group in opt.param_groups:group[\'lr\']=lr\n                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp_enabled):\n                    logits=model(src,tgt)\n                    loss=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=core.PAD)\n                if amp_enabled:\n                    scaler.scale(loss).backward();scaler.unscale_(opt)\n                    found=scaler._found_inf_per_device(opt)\n                    if len(found)!=1:raise RuntimeError(\'Expected one-device GradScaler state\')\n                    skipped=next(iter(found.values())).detach()\n                else:\n                    loss.backward();skipped=loss.new_zeros(())\n                norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n                finite=torch.isfinite(loss.detach()) & torch.isfinite(norm.detach())\n                records.append(torch.stack([loss.detach().float(),finite.float(),skipped.float()]))\n                if amp_enabled:scaler.step(opt);scaler.update()\n                else:opt.step()\n                counts.update(batch_counts);summary[\'steps_presented\']=j+1\n                del src,tgt,labels,logits,loss,norm\n                if stop_after is not None and j+1==stop_after:raise KeyboardInterrupt(\'CPU interruption fixture\')\n            core.sync(device);records=torch.stack(records).cpu().tolist()\n            summary[\'AMP_skips\']+=sum(int(x[2]>0) for x in records)\n            summary[\'optimizer_updates\']=summary[\'steps_presented\']-summary[\'AMP_skips\']\n            summary[\'finite_loss_and_grads\'] &= all(bool(x[1]) for x in records)\n            if not summary[\'finite_loss_and_grads\'] or summary[\'AMP_skips\']:\n                raise FloatingPointError(\'Nonfinite loss/grad or skipped AMP update; equal-update comparison invalid\')\n            if summary[\'steps_presented\']%log_every==0 or summary[\'steps_presented\']==len(plan):\n                elapsed=time.perf_counter()-train_start\n                point={\'update\':summary[\'steps_presented\'],\'lr\':lr,\'CE_last_chunk\':sum(x[0] for x in records)/len(records),\n                       \'seconds\':elapsed,\'pairs_per_second\':counts[\'pairs\']/elapsed,\n                       \'useful_target_tokens_per_second\':counts[\'target_tokens\']/elapsed}\n                history.append(point);print(config[\'id\'],json.dumps(point),flush=True)\n                core.write_json(outdir/\'progress.json\',{\'training\':summary,\'counts\':dict(counts),\'history\':history})\n        core.sync(device);train_end=time.perf_counter();summary[\'status\']=\'trained\'\n        summary[\'model_sha256\']=save_model_state(outdir/\'model.pt\',model)\n        summary[\'quality_eligible\']=True\n    except (torch.cuda.OutOfMemoryError,FloatingPointError) as error:\n        summary.update(status=\'invalid_training\',quality_eligible=False,error=str(error)[:500])\n    finally:\n        core.sync(device)\n        elapsed=(train_end or time.perf_counter())-train_start if train_start is not None else 0.\n        summary.update(train_seconds=elapsed,wall_seconds=time.perf_counter()-started,counts=dict(counts),history=history,\n                       pairs_per_second=counts[\'pairs\']/elapsed if elapsed else None,\n                       useful_source_tokens_per_second=counts[\'source_tokens\']/elapsed if elapsed else None,\n                       useful_target_tokens_per_second=counts[\'target_tokens\']/elapsed if elapsed else None,\n                       peak_allocated_bytes=torch.cuda.max_memory_allocated(device) if device.type==\'cuda\' else None,\n                       peak_reserved_bytes=torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None)\n        summary[\'time_fraction_of_saved22_training\']=elapsed/PROTOCOL[\'saved22_chain_training_seconds\']\n        if counts[\'source_slots\'] and counts[\'target_slots\']:\n            summary[\'padding_fraction\']=1-(counts[\'source_tokens\']+counts[\'target_tokens\'])/(counts[\'source_slots\']+counts[\'target_slots\'])\n        summary[\'history\']=history\n        if summary[\'status\']==\'trained\':\n            assert counts[\'pairs\']==plan.size and summary[\'optimizer_updates\']==len(plan)\n            core.write_json(outdir/\'training.json\',summary)\n        else:\n            core.write_json(outdir/\'incomplete-training.json\',summary)\n        del model,opt,scaler;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n    return summary\n\n@torch.inference_mode()\ndef ce_one(model_path,rows,tokenizer,device,spec=None,batch=32):\n    started=time.perf_counter();model=load_model(model_path,device,spec=spec).eval();total=torch.zeros((),device=device);denominator=0\n    cached=[([core.BOS]+core.encode(tokenizer,r[\'question\'])+[core.EOS],core.encode(tokenizer,r[\'code\'])) for r in rows]\n    order=sorted(range(len(rows)),key=lambda i:len(cached[i][0])+len(cached[i][1]))\n    try:\n        for lo in range(0,len(order),batch):\n            src,tgt,labels,counts=core.collate_cache(cached,order[lo:lo+batch],device)\n            logits=model(src,tgt)\n            total+=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=core.PAD,reduction=\'sum\')\n            denominator+=counts[\'target_tokens\']\n            del src,tgt,labels,logits\n        value=float(total.cpu())/denominator\n        if not math.isfinite(value):raise FloatingPointError(\'CE nonfinite\')\n        return {\'CE\':value,\'useful_target_tokens\':denominator,\'rows\':len(rows),\'precision\':\'FP32\',\'batch\':batch,\n                \'seconds\':time.perf_counter()-started}\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\n@torch.inference_mode()\ndef generate_one(model_path,rows,tokenizer_path,device,outdir,name,spec=None,batch=512,cap=512,half=True):\n    """Full timer begins before loading model tensors; ends at answers-file close."""\n    gc.collect()\n    if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n    start=time.perf_counter();model=load_model(model_path,device,spec=spec).eval()\n    tokenizer,tokenbytes=core.init_tokenizer(tokenizer_path)\n    if half:model.half()\n    try:\n        core.generate_preallocated(model,core.source_batch(tokenizer,[rows[0][\'question\']],device),cap=min(8,cap))\n        core.sync(device);generation_start=time.perf_counter()\n        encoded=[[core.BOS]+core.encode(tokenizer,r[\'question\'])+[core.EOS] for r in rows]\n        order=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i));ordered=[];steps_sum=row_steps_sum=0\n        for lo in range(0,len(order),batch):\n            indices=order[lo:lo+batch]\n            src=nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in indices],batch_first=True,padding_value=core.PAD).to(device)\n            tokens,steps=core.generate_preallocated(model,src,cap=cap)\n            steps_sum+=steps;row_steps_sum+=steps*len(indices)\n            for i,ids in zip(indices,tokens.cpu().tolist()):\n                active=core.active_tokens(ids)\n                ordered.append({\'id\':i,\'question\':rows[i][\'question\'],\'generated_token_ids\':active,\n                    \'generated_code\':core.decode_bytes(tokenbytes,active).decode(\'utf-8\',errors=\'replace\'),\n                    \'eos_emitted\':core.EOS in active})\n        examples=core.restore_order(ordered,order,len(rows));core.sync(device)\n        generation_seconds=time.perf_counter()-generation_start\n        outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n        answers=outdir/f\'{name}-answers.jsonl\'\n        with answers.open(\'w\',encoding=\'utf-8\') as f:\n            for e in examples:f.write(json.dumps({\'id\':e[\'id\'],\'question\':e[\'question\'],\'code\':e[\'generated_code\']},ensure_ascii=False)+\'\\n\')\n        full_seconds=time.perf_counter()-start\n        core.write_json(outdir/f\'{name}-generations.json\',examples)\n        timing={\'full_seconds\':full_seconds,\'generation_seconds\':generation_seconds,\'rows\':len(rows),\n                \'decode_steps_sum\':steps_sum,\'decode_row_steps_sum\':row_steps_sum,\n                \'answers_sha256\':core.sha_file(answers),\n                \'peak_reserved_bytes\':torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None,\n                \'timer\':\'model state load,restore,tokenizer,FP16/H2D,warmup,sorted greedy/order restore/answers write; download/cache/exec/CE/JSON dump outside\'}\n        timing[\'model_sha256\']=core.sha_file(model_path)\n        timing[\'tokenizer_sha256\']=core.sha_file(tokenizer_path)\n        return examples,timing\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef paired_mcnemar(score,control):\n    assert score[\'eligible_indices\']==control[\'eligible_indices\']\n    s,c=set(score[\'match_indices\']),set(control[\'match_indices\']);g,l=len(s-c),len(c-s);n=g+l\n    p=min(1.,(2*sum(math.comb(n,i) for i in range(min(g,l)+1)))/(1<<n)) if n else 1.\n    return {\'gains\':g,\'losses\':l,\'discordant\':n,\'exact_two_sided_p\':p,\n            \'delta_percentage_points\':100*(g-l)/score[\'denominator\']}\n\ndef length_accuracy(rows,tokenizer,score):\n    lengths=[len(core.encode(tokenizer,r[\'code\'])) for r in rows];eligible=set(score[\'eligible_indices\']);matched=set(score[\'match_indices\'])\n    groups={k:[] for k in [\'<60\',\'60–90\',\'90–120\',\'≥120\']}\n    for i,n in enumerate(lengths):\n        if i in eligible:groups[\'<60\' if n<60 else \'60–90\' if n<90 else \'90–120\' if n<120 else \'≥120\'].append(i)\n    return {k:{\'denominator\':len(ids),\'matches\':sum(i in matched for i in ids),\n               \'accuracy\':sum(i in matched for i in ids)/len(ids) if ids else None} for k,ids in groups.items()}\n\ndef evaluate_one(model_path,config,sets,refs,tokenizer_path,device,outdir,control=None,spec=None,\n                 gen_fn=generate_one,ce_fn=ce_one,execute_fn=numeric.evaluate_programs):\n    started=time.perf_counter();outdir=Path(outdir);metrics={\'id\':config[\'id\'],\'config\':config,\'status\':\'evaluation_started\',\'splits\':{}}\n    tokenizer,_=core.init_tokenizer(tokenizer_path)\n    for split,rows in sets.items():\n        gen_file=outdir/f\'{split}-generations.json\';time_file=outdir/f\'{split}-generation-time.json\'\n        if gen_file.exists() and time_file.exists():\n            examples=json.loads(gen_file.read_text());timing=json.loads(time_file.read_text())\n            assert len(examples)==len(rows) and all(a[\'question\']==b[\'question\'] for a,b in zip(examples,rows))\n            assert timing[\'model_sha256\']==core.sha_file(model_path) and timing[\'tokenizer_sha256\']==core.sha_file(tokenizer_path)\n        else:\n            examples,timing=gen_fn(model_path,rows,tokenizer_path,device,outdir,split,spec=spec)\n            core.write_json(time_file,timing)\n        execution=execute_fn([e[\'generated_code\'] for e in examples])\n        assert execution[\'summary\'][\'complete\']\n        core.write_json(outdir/f\'{split}-execution.json\',execution)\n        score=numeric.compare_exec(refs[split],execution)\n        assert score[\'denominator\']==(4096 if split==\'heldout4k\' else 991)\n        ce=ce_fn(model_path,rows,tokenizer,device,spec=spec)\n        m={\'exec\':score,\'CE\':ce,\'generation_time\':timing,\'length_accuracy\':length_accuracy(rows,tokenizer,score),\n           \'execution_seconds\':execution[\'summary\'][\'seconds\']}\n        if control:m[\'McNemar_against22\']=paired_mcnemar(score,control[\'splits\'][split][\'exec\'])\n        metrics[\'splits\'][split]=m\n        core.write_json(outdir/\'evaluation-progress.json\',metrics)\n        print(config[\'id\'],split,\'exec\',score[\'matches\'],\'/\',score[\'denominator\'],\'CE\',ce[\'CE\'],flush=True)\n    metrics[\'status\']=\'evaluated\';metrics[\'model_sha256\']=core.sha_file(model_path)\n    metrics[\'evaluation_wall_seconds_current_attempt\']=time.perf_counter()-started\n    core.write_json(outdir/\'metrics.json\',metrics)\n    return metrics\n\ndef best_run(metrics,ids=None):\n    candidates=[m for k,m in metrics.items() if (ids is None or k in ids) and m.get(\'status\')==\'evaluated\'\n                and m.get(\'quality_eligible\',True)]\n    if not candidates:raise RuntimeError(\'No complete eligible run; see saved progress/ZIP\')\n    return min(candidates,key=lambda m:(-m[\'splits\'][\'heldout4k\'][\'exec\'][\'matches\'],m[\'splits\'][\'heldout4k\'][\'CE\'][\'CE\'],m[\'id\']))\n\ndef needs_R6(lr_best):\n    s=lr_best[\'splits\'][\'old1024\'][\'exec\'];assert s[\'denominator\']==991\n    return s[\'matches\']<.18*s[\'denominator\']\n\ndef decisions(metrics,control,lr_best):\n    best=best_run(metrics);best128=best_run(metrics,[\'R1\',\'R2\',\'R3\',\'R4\',\'R5\'])\n    def acceptance(m):\n        old=m[\'splits\'][\'old1024\'][\'exec\'];new=m[\'splits\'][\'heldout4k\'][\'exec\'];baseline=control[\'splits\'][\'heldout4k\'][\'exec\']\n        return {\'id\':m[\'id\'],\'accepted\':old[\'matches\']>=.18*991 and new[\'matches\']-baseline[\'matches\']>=-.015*4096,\n                \'old991_accuracy\':old[\'accuracy\'],\'heldout4k_delta_pp\':100*(new[\'matches\']-baseline[\'matches\'])/4096}\n    ablations={}\n    for rid in [\'R4\',\'R5\']:\n        if rid not in metrics or metrics[rid].get(\'status\')!=\'evaluated\':ablations[rid]={\'status\':\'pending_or_invalid\'};continue\n        delta=metrics[rid][\'splits\'][\'heldout4k\'][\'exec\'][\'matches\']-lr_best[\'splits\'][\'heldout4k\'][\'exec\'][\'matches\']\n        ablations[rid]={\'accepted\':delta>=-.005*4096,\'clear_gain\':delta>=.01*4096,\n                         \'delta_pp\':100*delta/4096,\'McNemar_against_best_LR\':paired_mcnemar(\n                             metrics[rid][\'splits\'][\'heldout4k\'][\'exec\'],lr_best[\'splits\'][\'heldout4k\'][\'exec\'])}\n    return {\'best_LR_run\':lr_best[\'id\'],\'best_run\':acceptance(best),\'batch128_recipe\':acceptance(best128),\n            \'ablations\':ablations,\'R6_condition\':needs_R6(lr_best),\n            \'interpretation\':\'Acceptance thresholds are protocol rules, not statistical proof; timings measured separately. If only R6 passes, batch128 hypothesis is not confirmed.\'}\n\ndef pack_results(folder,archive):\n    folder=Path(folder);folder.mkdir(exist_ok=True)\n    files=[p for p in sorted(folder.rglob(\'*\')) if p.is_file() and p.name!=\'manifest.json\' and not p.name.endswith(\'.tmp\')]\n    manifest={\'experiment\':24,\'files\':[{\'file\':str(p.relative_to(folder)),\'bytes\':p.stat().st_size,\'sha256\':core.sha_file(p)} for p in files]}\n    core.write_json(folder/\'manifest.json\',manifest)\n    archive=Path(archive);pending=archive.with_suffix(\'.zip.tmp\')\n    # Model tensors stored without ZIP compression; JSON/NPY compressed lightly.\n    with zipfile.ZipFile(pending,\'w\') as z:\n        for p in files+[folder/\'manifest.json\']:\n            z.write(p,arcname=str(p.relative_to(folder)),compress_type=zipfile.ZIP_STORED if p.suffix==\'.pt\' else zipfile.ZIP_DEFLATED,compresslevel=1)\n    with zipfile.ZipFile(pending) as z:assert z.testzip() is None\n    pending.replace(archive)\n    return {\'name\':archive.name,\'bytes\':archive.stat().st_size,\'sha256\':core.sha_file(archive)}\n\ndef completed_training(outdir,config,initial_sha):\n    folder=Path(outdir);receipt=folder/\'training.json\';weights=folder/\'model.pt\'\n    if not receipt.exists():return None\n    row=json.loads(receipt.read_text())\n    assert row[\'config\']==config and row[\'initial_checkpoint_sha256\']==initial_sha\n    assert row[\'status\']==\'trained\' and row[\'quality_eligible\']\n    assert row[\'model_sha256\']==core.sha_file(weights)\n    assert row[\'steps_presented\']==config[\'updates\'] and row[\'counts\'][\'pairs\']==PAIR_PRESENTATIONS\n    return row\n\ndef launch(config,root,initial_path,cache,plan,sets,refs,tokenizer_path,device,control,fused,\n           train_fn=train_one,eval_fn=evaluate_one,pack_fn=pack_results):\n    folder=Path(root)/\'runs\'/config[\'id\'];folder.mkdir(parents=True,exist_ok=True)\n    receipt=completed_training(folder,config,core.sha_file(initial_path))\n    if receipt is None:\n        receipt=train_fn(config,initial_path,cache,plan,device,folder,fused=fused)\n        pack_fn(root,Path(root).with_suffix(\'.zip\'))\n        if not receipt.get(\'quality_eligible\'):\n            return {\'id\':config[\'id\'],\'status\':\'invalid_training\',\'quality_eligible\':False,\'training\':receipt}\n    metrics_file=folder/\'metrics.json\'\n    if metrics_file.exists():\n        metrics=json.loads(metrics_file.read_text())\n        assert metrics[\'config\']==config and metrics[\'model_sha256\']==core.sha_file(folder/\'model.pt\')\n    else:\n        metrics=eval_fn(folder/\'model.pt\',config,sets,refs,tokenizer_path,device,folder,control=control)\n    metrics.update(quality_eligible=True,training=receipt)\n    core.write_json(metrics_file,metrics)\n    pack_fn(root,Path(root).with_suffix(\'.zip\'))\n    return metrics\n', 'recipe24_input.py': '"""Hashed original21/22 input and safe boundary-resume of experiment24 bundles."""\nimport hashlib,json,shutil,zipfile\nfrom pathlib import Path,PurePosixPath\nimport torch\nimport recipe24_core as core\nimport recipe24_run as run\n\nTOKENIZER_SHA=\'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\'\nSOURCES={\n    \'22\':{\'zip\':\'4db40c0313f10afbf11d4a76beb2de59a729e1af9c6d40f08240c6f55697c615\',\n          \'pt\':{\'13f8ae4ae2ee2c15fc9d034cf8773de420226e073550c9774adb1846f3e27b23\',\n                \'5d115cf06c55f8d6245b5244d4cd622e38480fd5ef7dfe8806ba8bbcd70d4833\'}},\n    \'21\':{\'zip\':\'976670053c88a1f41d416595fc6ffb5be916ce7377cbb169cdfdeedac8a9c77b\',\n          \'pt\':{\'c06aac5ff2d879dbcfe8316086365a72f98d378937da02e69fb3cffcc66954ba\',\n                \'5e6a2740af730719bf3923bb2986309fa7835e0adb72978566e63711410c3bdb\'}}}\n\ndef allowed_resume_name(name):\n    p=PurePosixPath(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name:return False\n    root_names={\'protocol.json\',\'results.json\',\'decision.json\',\'manifest.json\',\'tokenizer.json\',\'initial.pt\',\n        \'initial.json\',\'batch-plan.npy\',\'plan.json\',\'heldout4k.json\',\'heldout4k-reference-execution.json\',\n        \'heldout4k-selection.json\',\'old1024.json\',\'old1024-reference-execution.json\',\'data.json\',\n        \'runtime.json\',\'test-generations.json\',\'test-answers.jsonl\',\'test-time.json\'}\n    if len(p.parts)==1:return name in root_names\n    if len(p.parts)==3 and p.parts[0] in {\'runs\',\'controls\'}:\n        ident=p.parts[1]\n        if ident not in {\'R1\',\'R2\',\'R3\',\'R4\',\'R5\',\'R6\',\'control22\',\'control21\'}:return False\n        return p.name in {\'model.pt\',\'source.json\',\'training.json\',\'incomplete-training.json\',\'progress.json\',\n            \'metrics.json\',\'evaluation-progress.json\',\'heldout4k-generations.json\',\'heldout4k-answers.jsonl\',\n            \'heldout4k-generation-time.json\',\'heldout4k-execution.json\',\'old1024-generations.json\',\n            \'old1024-answers.jsonl\',\'old1024-generation-time.json\',\'old1024-execution.json\'}\n    return False\n\ndef restore_bundle(path,destination,expected_code_sha):\n    destination=Path(destination)\n    with zipfile.ZipFile(path) as z:\n        if z.testzip() is not None:raise ValueError(\'Resume ZIP CRC failed\')\n        m=json.loads(z.read(\'manifest.json\'))\n        if m.get(\'experiment\')!=24:raise ValueError(\'Not experiment24 resume archive\')\n        entries=m[\'files\'];names=[x[\'file\'] for x in entries]\n        if len(names)>200 or len(names)!=len(set(names)):raise ValueError(\'Unexpected manifest\')\n        if len(z.namelist())!=len(names)+1 or set(z.namelist()) != set(names)|{\'manifest.json\'}:raise ValueError(\'ZIP/manifest mismatch\')\n        if any(not allowed_resume_name(n) for n in names):raise ValueError(\'Unsafe/unexpected resume member\')\n        if sum(z.getinfo(n).file_size for n in names)>1_500_000_000:raise ValueError(\'Oversized resume bundle\')\n        protocol=json.loads(z.read(\'protocol.json\'))\n        if protocol!={\'protocol\':run.PROTOCOL,\'code_sha256\':expected_code_sha}:\n            raise ValueError(\'Resume protocol/code differs; do not mix experiments\')\n        for x in entries:\n            if x[\'bytes\']>150_000_000:raise ValueError(\'Oversized member\')\n            raw=z.read(x[\'file\'])\n            if len(raw)!=x[\'bytes\'] or hashlib.sha256(raw).hexdigest()!=x[\'sha256\']:raise ValueError(\'Resume SHA mismatch\')\n            target=destination/x[\'file\']\n            if target.exists() and core.sha_file(target)!=x[\'sha256\']:\n                raise ValueError(\'Local file conflicts with uploaded resume; preserve both in separate sessions\')\n        for x in entries:\n            target=destination/x[\'file\'];target.parent.mkdir(parents=True,exist_ok=True)\n            if not target.exists():\n                with z.open(x[\'file\']) as source,target.open(\'wb\') as out:shutil.copyfileobj(source,out)\n    return {\'resume_archive_sha256\':core.sha_file(path),\'files_verified\':len(entries)}\n\ndef import_legacy_pt(path,source_id,root,digest):\n    assert digest in SOURCES[source_id][\'pt\'] and core.sha_file(path)==digest\n    folder=Path(root)/\'controls\'/f\'control{source_id}\';folder.mkdir(parents=True,exist_ok=True)\n    if (folder/\'model.pt\').exists():return\n    # Legacy checkpoint has Python/RNG metadata. Exact known SHA authorizes this load.\n    payload=torch.load(path,map_location=\'cpu\',weights_only=False)\n    assert payload[\'steps\']==100000 and payload[\'tokenizer_sha256\']==TOKENIZER_SHA\n    model=core.SimpleTransformer(**core.MODEL_SPEC);model.load_state_dict(payload[\'model\'],strict=True)\n    assert all(torch.isfinite(v).all() for v in model.state_dict().values())\n    tensor_digest=run.state_digest(model.state_dict())\n    saved=run.save_model_state(folder/\'model.pt\',model)\n    core.write_json(folder/\'source.json\',{\'source_experiment\':source_id,\'step\':100000,\'original_PT_sha256\':digest,\n        \'model_sha256\':saved,\'tensor_sha256\':tensor_digest,\'stored\':\'model state only; no optimizer\'})\n    del model,payload\n\ndef import_uploads(paths,root,code_sha):\n    root=Path(root);root.mkdir(exist_ok=True)\n    temp=Path(\'hw02-recipe24-input\');temp.mkdir(exist_ok=True)\n    paths=[Path(p) for p in paths]\n    report=[]\n    # Restore first; original uploads can then add optional control21.\n    originals=[]\n    for p in paths:\n        digest=core.sha_file(p)\n        original=next((i for i,s in SOURCES.items() if digest==s[\'zip\']),None)\n        if p.suffix.lower()==\'.zip\' and original is None:\n            report.append(restore_bundle(p,root,code_sha))\n        else:originals.append((p,digest,original))\n    for p,digest,source_id in originals:\n        if source_id:\n            with zipfile.ZipFile(p) as z:\n                found=False\n                for info in z.infolist():\n                    name=PurePosixPath(info.filename).name\n                    if name!=\'tokenizer.json\' and not (name==\'latest.pt\' or name.endswith(\'100000.pt\')):continue\n                    if info.file_size>150_000_000:continue\n                    raw=z.read(info.filename);s=hashlib.sha256(raw).hexdigest()\n                    if s==TOKENIZER_SHA:\n                        (root/\'tokenizer.json\').write_bytes(raw)\n                    elif s in SOURCES[source_id][\'pt\'] and not found:\n                        local=temp/f\'source{source_id}.pt\';local.write_bytes(raw)\n                        import_legacy_pt(local,source_id,root,s);found=True\n                if not found:raise ValueError(\'Original ZIP has no verified100k checkpoint\')\n            report.append({\'original_archive\':source_id,\'sha256\':digest})\n        elif digest==TOKENIZER_SHA:\n            shutil.copyfile(p,root/\'tokenizer.json\')\n            report.append({\'kind\':\'tokenizer\',\'sha256\':digest,\'verified_before_data\':True})\n        else:\n            source_id=next((i for i,s in SOURCES.items() if digest in s[\'pt\']),None)\n            if source_id is None:raise ValueError(f\'Unexpected upload or SHA mismatch: {p.name}\')\n            import_legacy_pt(p,source_id,root,digest)\n            report.append({\'kind\':\'checkpoint\',\'source_experiment\':source_id,\'sha256\':digest,\'verified_before_weights_and_data\':True})\n    token=root/\'tokenizer.json\'\n    if not token.exists() or core.sha_file(token)!=TOKENIZER_SHA:raise ValueError(\'Need verified BPE8k tokenizer22\')\n    if not (root/\'controls/control22/model.pt\').exists():raise ValueError(\'Need22/100k checkpoint or a complete boundary-resume ZIP24\')\n    protocol={\'protocol\':run.PROTOCOL,\'code_sha256\':code_sha}\n    if (root/\'protocol.json\').exists():assert json.loads((root/\'protocol.json\').read_text())==protocol\n    else:core.write_json(root/\'protocol.json\',protocol)\n    return report\n'}
for name,source in MODULE_SOURCES.items():
    (MODULE_DIR/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(MODULE_DIR.resolve()))
import recipe24_core as core,recipe24_exec as numeric,recipe24_data as data,recipe24_run as recipe,recipe24_input as inputs
for module in (core,numeric,data,recipe,inputs):importlib.reload(module)
import torch,numpy as np,tokenizers,pyarrow,sympy
assert torch.cuda.is_available() and torch.cuda.device_count()==1,'Нужна однаGPU'
assert 'T4' in torch.cuda.get_device_name(0),'Протокол рассчитан наT4'
assert tokenizers.__version__=='0.23.2'
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
torch.set_float32_matmul_precision('highest')
device=torch.device('cuda')
CODE_SHA=hashlib.sha256(json.dumps(MODULE_SOURCES,sort_keys=True).encode()).hexdigest()
RUN_DIR=Path('hw02-fast-recipe-equal-presentations-lr-and-ablations-results')
RUN_DIR.mkdir(exist_ok=True)
ARCHIVE=RUN_DIR.with_suffix('.zip')
FUSED=core.fused_adamw_available(device)
runtime={'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'CUDA':torch.version.cuda,
         'tokenizers':tokenizers.__version__,'pyarrow':pyarrow.__version__,'sympy':sympy.__version__,
         'SDPA_enabled_flags':core.sdpa_flags(),'fusedAdamW_available':FUSED,'TF32':False}
print('RUNTIME',json.dumps(runtime,ensure_ascii=False))
print(subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total,memory.free','--format=csv'],capture_output=True,text=True).stdout)
with torch.device('meta'):parameter_model=core.SimpleTransformer(**core.MODEL_SPEC)
print('PARAMETERS',sum(p.numel() for p in parameter_model.parameters()));del parameter_model
def persist():
    result['elapsed_session_since_first_cell']=time.monotonic()-_RECIPE24_FIRST_CELL
    core.write_json(RUN_DIR/'results.json',result)
def snapshot():
    persist();print('ZIP',recipe.pack_results(RUN_DIR,ARCHIVE),flush=True)

In [ ]:
import shutil

# auto | drive | paths | upload | reuse
INPUT_MODE = 'auto'
INPUT_PATHS = ['/content/drive/MyDrive/nlp-hw02/24-start-from-22-model-only.zip']
COMPACT_INPUT_NAME = '24-start-from-22-model-only.zip'
COMPACT_INPUT_SHA = '5b70c8adebc8affaa20b2aa7254fbfc5eb048c0371b0ad286130fbed2cd85de8'

def acquire_recipe24_inputs(mode, paths, root, upload_dir, code_sha):
    root, upload_dir = Path(root), Path(upload_dir)
    if mode not in {'auto', 'drive', 'paths', 'upload', 'reuse'}:
        raise ValueError('INPUT_MODE: auto, drive, paths, upload or reuse')
    required = ['tokenizer.json', 'protocol.json', 'controls/control22/model.pt', 'controls/control22/source.json']
    if mode == 'auto':
        mode = 'reuse' if all((root/name).is_file() for name in required) else 'upload'
    upload_dir.mkdir(parents=True, exist_ok=True)
    selected, report = [], []
    started = time.monotonic()
    if mode == 'drive':
        from google.colab import drive
        drive.mount('/content/drive')
    if mode in {'drive', 'paths'}:
        if not paths:
            raise ValueError('Укажите INPUT_PATHS')
        for index, name in enumerate(paths):
            source = Path(name)
            if not source.is_file():
                raise FileNotFoundError(f'Входной файл не найден: {source}')
            digest = core.sha_file(source)
            if source.name == COMPACT_INPUT_NAME and digest != COMPACT_INPUT_SHA:
                raise ValueError('Compact starter ZIP SHA mismatch')
            local = upload_dir/f'{index}-{source.name}'
            if source.resolve() != local.resolve():
                if not local.exists() or core.sha_file(local) != digest:
                    pending = local.with_suffix(local.suffix+'.tmp')
                    shutil.copyfile(source, pending)
                    if core.sha_file(pending) != digest:
                        raise ValueError('Input copy SHA mismatch')
                    pending.replace(local)
            selected.append(local)
            report.append({'input_file':source.name, 'bytes':source.stat().st_size, 'sha256':digest})
    elif mode == 'upload':
        from google.colab import files
        uploaded = files.upload()
        for name, raw in uploaded.items():
            local = upload_dir/Path(name).name
            if local.name == COMPACT_INPUT_NAME and hashlib.sha256(raw).hexdigest() != COMPACT_INPUT_SHA:
                raise ValueError('Compact starter ZIP SHA mismatch')
            local.write_bytes(raw)
            selected.append(local)
        del uploaded
    report.extend(inputs.import_uploads(selected, root, code_sha))
    # Verify retained controls before dataset preparation, also on auto/reuse.
    for ident in ('22', '21'):
        folder = root/'controls'/f'control{ident}'
        if not folder.exists():
            continue
        source = json.loads((folder/'source.json').read_text())
        if (source['source_experiment'] != ident or source['step'] != 100000
            or source['original_PT_sha256'] not in inputs.SOURCES[ident]['pt']
            or source['model_sha256'] != core.sha_file(folder/'model.pt')):
            raise ValueError(f'Control{ident} provenance/model SHA mismatch')
    report.append({'input_mode':mode, 'input_seconds':time.monotonic()-started})
    return report

upload_report=acquire_recipe24_inputs(INPUT_MODE, INPUT_PATHS, RUN_DIR, Path('hw02-recipe24-upload'), CODE_SHA)
TOKENIZER_PATH=RUN_DIR/'tokenizer.json'
result=json.loads((RUN_DIR/'results.json').read_text()) if (RUN_DIR/'results.json').exists() else {
    'experiment':24,'protocol':recipe.PROTOCOL,'status':'inputs_verified','runs':{},'controls':{},'sessions':[]}
assert result['experiment']==24
result['sessions'].append({'runtime':runtime,'uploads':upload_report})
if 'fused_adamw' in result:
    assert result['fused_adamw']==FUSED,'Optimizer mode changed across resume; preserve the common recipe'
else:result['fused_adamw']=FUSED
core.write_json(RUN_DIR/'runtime.json',result['sessions'])
SESSION_ACTIVE_STARTED=time.monotonic()
persist()
print('INPUTS VERIFIED BEFORE DATA',upload_report)

## Фиксированные данные и план

Перед обучением проверяются pinned SHA шардов/tokenizer/train500k/old1024, SHA новогоheldout4k и плана. Shards3–5 используются только для оценки; первых4096 подходящих строк оказалось достаточно вshard3. Reference-фильтр применяется к новомуheldout, train не меняется.

CE считается FP32 на всех reference-target токенах, включаяEOS и исключаяPAD. Для old1024 CE охватывает1024 строки, exec — только991 числовой reference. Length bins используют BPE-токены программы **безEOS**, интервалы[0,60),[60,90),[90,120),[120,∞).

In [ ]:
try:
    result['status']='data_preparation';persist()
    tokenizer,tokenbytes=core.init_tokenizer(TOKENIZER_PATH)
    cache,old,test,paths,info,fresh_questions=data.prepare_data('hw02-data',tokenizer)
    forbidden={r['question'] for r in old+test}|fresh_questions
    if (RUN_DIR/'heldout4k.json').exists():
        new=json.loads((RUN_DIR/'heldout4k.json').read_text())
        selection=json.loads((RUN_DIR/'heldout4k-selection.json').read_text())
    else:
        new,new_execution,selection=data.prepare_heldout4k('hw02-data',forbidden,numeric.evaluate_programs)
        assert data.data_hash(new)==recipe.PROTOCOL['heldout4k_pairs_sha256']
        core.write_json(RUN_DIR/'heldout4k-selection.json',selection)
        core.write_json(RUN_DIR/'heldout4k.json',new)
    assert len(new)==len({r['question'] for r in new})==4096
    assert data.data_hash(new)==recipe.PROTOCOL['heldout4k_pairs_sha256']
    assert not ({r['question'] for r in new}&forbidden)
    assert all(data.eligible(r) and data.is_validation(r['question']) for r in new)
    del fresh_questions,forbidden
    sets={'heldout4k':new,'old1024':old}
    refs={}
    for split,rows in sets.items():
        refs[split]=numeric.evaluate_programs([r['code'] for r in rows])
        expected=4096 if split=='heldout4k' else 991
        assert refs[split]['summary']['complete'] and refs[split]['summary']['statuses'].get('numeric')==expected
        core.write_json(RUN_DIR/f'{split}-reference-execution.json',refs[split])
        print('REFERENCE',split,refs[split]['summary'])
    core.write_json(RUN_DIR/'old1024.json',old)
    if (RUN_DIR/'batch-plan.npy').exists():
        plan=np.load(RUN_DIR/'batch-plan.npy',allow_pickle=False)
    else:
        plan=np.asarray(core.bucket_plan(cache,128,6250,seed=recipe.PLAN_SEED),dtype=np.int32)
        np.save(RUN_DIR/'batch-plan.npy',plan,allow_pickle=False)
    assert plan.shape==(6250,128) and plan.dtype==np.int32 and int(plan.min())>=0 and int(plan.max())<500000
    assert core.sha_file(RUN_DIR/'batch-plan.npy')==recipe.PROTOCOL['batch_plan_file_sha256']
    source_lengths=np.array([len(x[0]) for x in cache]);target_lengths=np.array([len(x[1])+1 for x in cache])
    counts={'pairs':int(plan.size),'source_tokens':int(source_lengths[plan].sum()),'target_tokens':int(target_lengths[plan].sum())}
    assert counts['pairs']==800000 and counts['target_tokens']==52797347 and counts['source_tokens']==38706084
    core.write_json(RUN_DIR/'plan.json',{'SHA256':core.sha_file(RUN_DIR/'batch-plan.npy'),'counts':counts,
        'R6_same_flat_plan':True,'seed':recipe.PLAN_SEED,'chunk_pairs':4096})
    INITIAL=RUN_DIR/'initial.pt'
    if not INITIAL.exists():
        digest=recipe.make_initial(INITIAL)
        core.write_json(RUN_DIR/'initial.json',{'tensor_SHA256':digest,'file_SHA256':core.sha_file(INITIAL),'seed':42})
    initial=json.loads((RUN_DIR/'initial.json').read_text())
    assert initial['file_SHA256']==core.sha_file(INITIAL)
    info.update(heldout4k_pairs_sha256=data.data_hash(new),heldout4k_file_sha256=core.sha_file(RUN_DIR/'heldout4k.json'),
                heldout4k_examples=4096,counts_per_launch=counts)
    result['data']=info;core.write_json(RUN_DIR/'data.json',info)
    result['status']='data_ready';snapshot()
    print('PAIR PRESENTATIONS AND USEFUL TOKENS',counts)
except BaseException as error:
    result['status']='setup_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## Контроль22 и необязательный21

Все модели оцениваются одним FP16 sorted batch512 greedy-путём. Здесь контроль22 может дать190/991, как FP16/b512 в23; прежние189/991 относились кFP32. Результат22 обязан попасть в190±3; иначе обучение не начинается. Контроль21 оценивается, если его веса предоставлены. Каждый run получает McNemar против **текущего** контроля22 на каждом split.

In [ ]:
try:
    result['status']='controls';persist()
    for cid in ['control22','control21']:
        folder=RUN_DIR/'controls'/cid
        if not (folder/'model.pt').exists():
            print(cid,'not provided');continue
        if (folder/'metrics.json').exists():
            m=json.loads((folder/'metrics.json').read_text())
            assert m['model_sha256']==core.sha_file(folder/'model.pt')
        else:
            config={'id':cid,'source_step':100000,'precision':'FP16 sorted b512 greedy cap512'}
            if cid=='control22':
                model=recipe.load_model(folder/'model.pt',device).eval().half()
                src=core.source_batch(tokenizer,[old[0]['question']],device)
                result['observed_SDPA_control']=core.profile_sdpa(lambda:core.generate_preallocated(model,src,8),device)
                print('OBSERVED SDPA',result['observed_SDPA_control']);del model,src;gc.collect();torch.cuda.empty_cache()
            m=recipe.evaluate_one(folder/'model.pt',config,sets,refs,TOKENIZER_PATH,device,folder,
                 control=result['controls'].get('control22') if cid=='control21' else None)
        result['controls'][cid]=m;snapshot()
    CONTROL=result['controls']['control22']
    assert 187<=CONTROL['splits']['old1024']['exec']['matches']<=193,'Control22 differs from23; stop before training'
    result['status']='controls_passed';persist()
except BaseException as error:
    result['status']='control_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## R1–R3: выбор LR

Каждый запуск начинается с одногоinitial-state и того же плана. После полногоtrain сохраняются только model tensors; послеevaluation — генерации/exec/CE/length bins/McNemar. ZIP обновляется на обеих границах. Полный повтор ячейки пропускает уже завершённые этапы.

In [ ]:
try:
    for rid,lr in zip(['R1','R2','R3'],recipe.PROTOCOL['peak_LRs']):
        result['status']='running_'+rid;persist()
        config=recipe.run_spec(rid,lr)
        result['runs'][rid]=recipe.launch(config,RUN_DIR,INITIAL,cache,plan,sets,refs,TOKENIZER_PATH,device,CONTROL,FUSED)
        snapshot()
    LR_BEST=recipe.best_run(result['runs'],['R1','R2','R3'])
    BEST_LR=LR_BEST['config']['peak_lr']
    result['best_LR_run']=LR_BEST['id'];result['best_peak_LR']=BEST_LR
    result['R6_required']=recipe.needs_R6(LR_BEST)
    result['status']='LR_selected';snapshot()
    print('BEST LR',LR_BEST['id'],BEST_LR,'R6_REQUIRED',result['R6_required'])
except BaseException as error:
    result['status']='LR_search_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## R4/R5 и условный R6

R4 иR5 — независимые абляции лучшегоLR: изменения не объединяются. R5 сохраняетdropout.1; исходные token draws делятся на√d, все другие начальные tensors совпадают. Это даётstd≈1 после прежнего множителя√d; позиции не меняются.

R6 запускается по результату лучшего **R1–R3**, даже если абляция позже улучшит old991. План reshaped128→32 сохраняет весь порядок800k пар и точные useful-token counts. Обновлений становится25000, warmup1250; peakLR вдвое меньше.

In [ ]:
try:
    LR_BEST=recipe.best_run(result['runs'],['R1','R2','R3']);BEST_LR=LR_BEST['config']['peak_lr']
    configs=[recipe.run_spec('R4',BEST_LR,dropout=0.),recipe.run_spec('R5',BEST_LR,scaled=True)]
    if recipe.needs_R6(LR_BEST):configs.append(recipe.run_spec('R6',BEST_LR/2,batch=32))
    else:print('R6 not required: best R1–R3 reached18% on old991')
    for config in configs:
        result['status']='running_'+config['id'];persist()
        result['runs'][config['id']]=recipe.launch(config,RUN_DIR,INITIAL,cache,plan,sets,refs,TOKENIZER_PATH,device,CONTROL,FUSED)
        snapshot()
    result['status']='runs_evaluated';snapshot()
except BaseException as error:
    result['status']='ablations_or_R6_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## Решение и test1919

Основной выбор — exec4k, CE разрешает равенство. Пороговые правила принятия не заменяют статистическое свидетельство: отдельно выводятся paired McNemar и bins. Для лучшего запуска формируетсяtest1919 JSONL в исходном порядке с полным таймером. Gold test недоступен; форматJSONL внутренний, точная схема посылки бота остаётся неизвестной.

In [ ]:
try:
    LR_BEST=recipe.best_run(result['runs'],['R1','R2','R3'])
    decision=recipe.decisions(result['runs'],CONTROL,LR_BEST)
    BEST=recipe.best_run(result['runs'])
    result['decision']=decision;core.write_json(RUN_DIR/'decision.json',decision)
    table=[]
    for rid,m in result['runs'].items():
        if m.get('status')!='evaluated':print('INCOMPLETE',rid,m.get('status'));continue
        t=m['training'];s=m['splits']
        table.append({'id':rid,'peak_LR':m['config']['peak_lr'],'batch':m['config']['batch'],
            'pairs':t['counts']['pairs'],'target_tokens':t['counts']['target_tokens'],'train_min':t['train_seconds']/60,
            'time_fraction_of_saved22':t['time_fraction_of_saved22_training'],
            'pairs/s':t['pairs_per_second'],'target_tok/s':t['useful_target_tokens_per_second'],
            '4k_matches':s['heldout4k']['exec']['matches'],'4k_accuracy':s['heldout4k']['exec']['accuracy'],
            'old991_matches':s['old1024']['exec']['matches'],'CE4k':s['heldout4k']['CE']['CE'],
            'McNemar_vs22':s['heldout4k']['McNemar_against22']['exact_two_sided_p']})
    import pandas as pd
    display(pd.DataFrame(table))
    print('DECISION',json.dumps(decision,ensure_ascii=False,indent=2))
    for rid,m in result['runs'].items():
        if m.get('status')=='evaluated':print('LENGTH BINS',rid,json.dumps(m['splits']['heldout4k']['length_accuracy'],ensure_ascii=False))
    best_model=RUN_DIR/'runs'/BEST['id']/'model.pt'
    if (RUN_DIR/'test-time.json').exists():
        test_time=json.loads((RUN_DIR/'test-time.json').read_text())
        assert test_time['run']==BEST['id'] and test_time['model_SHA256']==core.sha_file(best_model)
    else:
        examples,test_time=recipe.generate_one(best_model,test,TOKENIZER_PATH,device,RUN_DIR,'test')
        assert [e['question'] for e in examples]==[r['question'] for r in test]
        test_time.update(run=BEST['id'],model_SHA256=core.sha_file(best_model),under180_seconds=test_time['full_seconds']<180)
        core.write_json(RUN_DIR/'test-time.json',test_time)
    print('TEST FULL WALL',test_time)
    result['test']=test_time;result['table']=table;result['status']='complete'
    result['session_active_wall_seconds']=time.monotonic()-SESSION_ACTIVE_STARTED
    result['target_budget_minutes']=115 if recipe.needs_R6(LR_BEST) else 90
    result['within_target_session_budget']=result['session_active_wall_seconds']<=result['target_budget_minutes']*60
    snapshot()
except BaseException as error:
    result['status']='report_or_test_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## Сохранить текущий ZIP — работает после прерывания

Ячейка самостоятельная: не обучает и не генерирует. В архив входят model-state PT, общий initial-state, controls, tokenizer/план/heldout, генерации/метрики/progress и manifest с SHA каждого файла. Данные train/optimizer/scaler не включаются.

Для сохранения в Google Drive поставить `SAVE_TO_DRIVE=True`; `DRIVE_RESULTS_DIR` задаёт папку. Можно поставить `DOWNLOAD_ZIP=False`, чтобы обойти скачивание через браузер. При следующем runtime указать сохранённый ZIP в `INPUT_PATHS` и выбрать `INPUT_MODE='drive'`: завершённые train/evaluation будут переиспользованы. Копия в Drive создаётся при выполнении этой ячейки; до её выполнения результаты находятся на диске runtime.

Проверено локально на CPU: новый heldout/план, tiny train/full–KV/порядок/resume/архив и правила; компактный ZIP, тождественность весов22, пути/reuse и копирование проверены отдельно. Полные R1–R6/AMP-T4/большой batch/90–115мин/качество, реальная скорость передачи и Colab/Drive UI до пользовательского запуска не проверены.


In [ ]:
import json,hashlib,zipfile
from pathlib import Path
# Сохранение в Drive вызывается только при SAVE_TO_DRIVE=True.
SAVE_TO_DRIVE = False
DOWNLOAD_ZIP = True
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/nlp-hw02/results'
folder=Path('hw02-fast-recipe-equal-presentations-lr-and-ablations-results');folder.mkdir(exist_ok=True)
if not (folder/'results.json').exists():
    (folder/'results.json').write_text(json.dumps({'experiment':24,'status':'partial_before_setup'}),encoding='utf-8')
def file_sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024**2),b''):h.update(b)
    return h.hexdigest()
paths=[p for p in sorted(folder.rglob('*')) if p.is_file() and p.name!='manifest.json' and not p.name.endswith('.tmp')]
manifest={'experiment':24,'files':[{'file':str(p.relative_to(folder)),'bytes':p.stat().st_size,'sha256':file_sha(p)} for p in paths]}
(folder/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
archive=folder.with_suffix('.zip');pending=archive.with_suffix('.zip.tmp')
with zipfile.ZipFile(pending,'w') as z:
    for p in paths+[folder/'manifest.json']:
        z.write(p,arcname=str(p.relative_to(folder)),compress_type=zipfile.ZIP_STORED if p.suffix=='.pt' else zipfile.ZIP_DEFLATED,compresslevel=1)
with zipfile.ZipFile(pending) as z:
    assert z.testzip() is None
    for e in manifest['files']:
        with z.open(e['file']) as f:
            h=hashlib.sha256()
            for b in iter(lambda:f.read(1024**2),b''):h.update(b)
        assert h.hexdigest()==e['sha256']
pending.replace(archive)
print('ZIP',archive.name,archive.stat().st_size,'bytes;status',json.loads((folder/'results.json').read_text()).get('status'))
if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    destination=Path(DRIVE_RESULTS_DIR);destination.mkdir(parents=True,exist_ok=True)
    target=destination/archive.name
    if target.resolve()!=archive.resolve():
        pending_copy=target.with_suffix('.zip.tmp')
        shutil.copyfile(archive,pending_copy)
        assert file_sha(pending_copy)==file_sha(archive),'Drive copy SHA mismatch; local ZIP is available'
        pending_copy.replace(target)
    print('DRIVE ZIP',str(target),'; SHA256',file_sha(target))
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive))
